# Filling the Missing Square: Inpainting on AFHQ
Setup, data, fixed test masks and shared metrics. Methods 1-6 get their own sections at the bottom.

Runs on Colab / Kaggle / local. On Colab set a GPU runtime first.

## 0. Setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/ericacleo/dlpastia.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm
print(os.getcwd())

## 1. Download data (Kaggle)
Needs Kaggle credentials: on Colab set the env vars below or upload `kaggle.json`; on Kaggle notebooks use *Add Input* with `andrewmvd/animal-faces` instead.

In [ ]:
import os, yaml, kagglehub
from pathlib import Path
cfg = yaml.safe_load(open('configs/data.yaml'))
# os.environ['KAGGLE_USERNAME'] = '...'; os.environ['KAGGLE_KEY'] = '...'
DATA = Path(kagglehub.dataset_download(cfg['kaggle_dataset']))
print(DATA)
for p in sorted(DATA.rglob('*')):
    if p.is_dir(): print(p.relative_to(DATA), len(list(p.glob('*'))))

## 2. Dataset + fixed masks
Train masks are random each time. Test masks are fixed by seed and shared by all methods.

In [ ]:
import numpy as np, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from src.masks import random_square_mask, composite

SIZE = cfg['image_size']
class AFHQ(Dataset):
    def __init__(self, files, train=True, seed=cfg['test_seed']):
        self.files, self.train, self.seed = files, train, seed
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB').resize((SIZE, SIZE), Image.BICUBIC)
        x = torch.from_numpy(np.asarray(img).copy()).permute(2,0,1).float()/127.5 - 1   # [-1,1]
        rng = np.random.default_rng(None if self.train else self.seed + i)
        m = torch.from_numpy(random_square_mask(SIZE, cfg['mask_ratio'], rng))[None]
        return x, m

# Adjust after checking the folder layout printed above (train/val, cat/dog/wild)
all_files = sorted(DATA.rglob('*.jpg')) + sorted(DATA.rglob('*.png'))
train_files = [f for f in all_files if 'train' in f.parts]
test_files  = [f for f in all_files if ('val' in f.parts or 'test' in f.parts)]
print(len(train_files), len(test_files))
train_ds, test_ds = AFHQ(train_files, True), AFHQ(test_files, False)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 6, figsize=(14,5))
for k in range(6):
    x, m = test_ds[k]
    ax[0,k].imshow(((x+1)/2).permute(1,2,0)); ax[0,k].axis('off')
    ax[1,k].imshow((((1-m)*x+1)/2).permute(1,2,0)); ax[1,k].axis('off')
plt.show()

## 3. Shared metrics (hole region)
PSNR, SSIM, LPIPS on the hole; FID over the whole test set; diversity = mean pairwise LPIPS across K fills.

In [ ]:
import lpips
from skimage.metrics import structural_similarity as ssim_fn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
lp = lpips.LPIPS(net='alex').to(dev)

def hole_metrics(pred, gt, m):
    # pred, gt: (B,3,H,W) in [-1,1]; m: (B,1,H,W)
    out = composite(pred, gt, m)
    mse = (((out-gt)**2*m).sum((1,2,3)) / (3*m.sum((1,2,3)))).clamp_min(1e-10)
    psnr = 10*torch.log10(4.0/mse)                      # range is 2 -> MAX^2 = 4
    lpv = lp((out*m).to(dev), (gt*m).to(dev)).flatten().cpu()
    ss = []
    for o, g in zip(out, gt):
        a, b = ((o+1)/2).permute(1,2,0).numpy(), ((g+1)/2).permute(1,2,0).numpy()
        ss.append(ssim_fn(a, b, channel_axis=2, data_range=1.0))
    return dict(psnr=psnr.mean().item(), ssim=float(np.mean(ss)), lpips=lpv.mean().item())

def diversity(fills):  # list of K tensors (B,3,H,W)
    d = [lp(fills[i].to(dev), fills[j].to(dev)).mean().item()
         for i in range(len(fills)) for j in range(i+1, len(fills))]
    return float(np.mean(d))

# FID: save composited outputs per method, then
# torch_fidelity.calculate_metrics(input1=<out_dir>, input2=<real_test_dir>, fid=True)

x, m = next(iter(DataLoader(test_ds, batch_size=8)))
print(hole_metrics(x, x, m))                  # perfect fill sanity check
print(hole_metrics(torch.zeros_like(x), x, m))  # grey-fill baseline

## 4. Methods
Each method should expose `fill(x_masked, mask, K) -> list of K tensors`, then be scored with section 3.

### 4.1 MAE (self-supervised)

In [ ]:
# TODO: pretrained MAE (ViT) + fine-tune on AFHQ; hide patches overlapping the hole

### 4.2 StyleGAN inversion (unsupervised)

In [ ]:
# TODO: StyleGAN2-ADA AFHQ; optimise w on visible pixels (MSE + LPIPS)

### 4.3 StyleGAN non-inversion (supervised, CoModGAN-style)

In [ ]:
# TODO: encoder(masked img + mask) + z -> StyleGAN generator; adv + R1 (+ LPIPS)

### 4.4 Mirror-conditioned StyleGAN (supervised)

In [ ]:
# TODO: same as 4.3 with flipped masked image + flipped mask (8 channels); optional symmetry loss

### 4.5 RePaint (unsupervised diffusion)

In [ ]:
# TODO: pretrained/trained unconditional DDPM on AFHQ + RePaint sampler

### 4.6 Mirror-conditioned + diffusion (ours)
Decision pending: (a) trained diffusion with mirror input, or (b) training-free adaptive mirror guidance on RePaint.

In [ ]:
# TODO

## 5. Results table + qualitative grids

In [ ]:
# TODO: collect metrics for all methods, K=5 fills, same test masks